In [5]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from baseline import build_dictionary, translate
from score import chrf, bleu

train_data = pd.read_csv("data/train/data.csv")
train_answers = pd.read_csv("data/train/answers.csv")

train = train_data.merge(train_answers , on = "id")

print(train.head())
print(train.shape)


         id                                                src  \
0  tr_00000             mawohu husade razata demo Ningu sagema   
1  tr_00001   hero napuwode mudahe kiboni wupu nikori bazobesu   
2  tr_00002                             Zebogu husade resomata   
3  tr_00003  zado Nongori pituhota kara nemuke rohoyu nitok...   
4  tr_00004                                    beyuti nubepusu   

                                         translation  
0  con ngựa đã bơi ở bờ sông nhưng Ningu không là...  
1  bà cụ sẽ tặng chiếc mũ mới cho mẹ trẻ ở bệnh viện  
2                       Zebogu chưa câu cá ở bờ sông  
3  ngày mai em bé sẽ đọc những lá thư rất dài ở c...  
4                                chúng tôi sẽ đi dạo  
(20000, 3)


In [6]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(train, groups=train["src"])
)

train_fit = train.iloc[train_idx].reset_index(drop=True)
val = train.iloc[val_idx].reset_index(drop=True)

print("Train:", train_fit.shape)
print("Val:", val.shape)

overlap = set(train_fit["src"]) & set(val["src"])
print("SRC overlap:", len(overlap))

Train: (16019, 3)
Val: (3981, 3)
SRC overlap: 0


In [8]:
lex_base = build_dictionary(
    train_fit["src"].tolist(),
    train_fit["translation"].tolist()
)

pred_base = [
    translate(s, lex_base)
    for s in val["src"]
]

base_chrf = chrf(
    pred_base,
    val["translation"].tolist()
)

base_bleu = bleu(
    pred_base,
    val["translation"].tolist()
)

print("Baseline chrF:", base_chrf)
print("Baseline BLEU:", base_bleu)

Baseline chrF: 44.99368551070516
Baseline BLEU: 9.363060504659252


In [9]:
ratios = [
    0.60,
    0.70,
    0.80,
    0.85,
    0.90,
    0.95,
    1.00
]

results = []

for ratio in ratios:

    lex_tmp = build_dictionary(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=ratio
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    score_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    results.append({
        "ratio": ratio,
        "val_chrf": score_tmp
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "val_chrf",
    ascending=False
)

,ratio,val_chrf
0,0.60,51.898131
1,0.70,50.531631
2,0.80,45.675351
3,0.85,45.066728
4,0.90,44.993686
5,0.95,44.748730
6,1.00,44.017845


In [11]:
ratios_fine = [
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.625,
    0.65,
    0.675,
    0.70
]

results_fine = []

for ratio in ratios_fine:
    lex_tmp = build_dictionary(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=ratio
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    score_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    results_fine.append({
        "ratio": ratio,
        "val_chrf": score_tmp
    })

pd.DataFrame(results_fine).sort_values(
    "val_chrf",
    ascending=False
)

,ratio,val_chrf
5,0.625,52.457132
4,0.600,51.898131
6,0.650,51.294121
2,0.500,51.269776
7,0.675,50.835378
8,0.700,50.531631
3,0.550,50.518097
1,0.450,50.393316
0,0.400,49.849609


In [13]:
from collections import Counter, defaultdict

def build_dictionary_v2(
    src,
    tgt,
    ratio=0.625,
    top_k=3
):
    src_cnt = Counter()
    tgt_cnt = Counter()
    pair_cnt = defaultdict(Counter)
    pos_sum = defaultdict(float)

    for s, t in zip(src, tgt):
        stoks = set(s.split())
        ttoks = t.split()

        for i, w in enumerate(ttoks):
            pos_sum[w] += i / max(len(ttoks) - 1, 1)

        tset = set(ttoks)

        src_cnt.update(stoks)
        tgt_cnt.update(tset)

        for a in stoks:
            pair_cnt[a].update(tset)

    tgt_freq = Counter(
        w
        for t in tgt
        for w in t.split()
    )

    avg_pos = {
        w: pos_sum[w] / tgt_freq[w]
        for w in tgt_freq
    }

    lex = {}

    for a, cnts in pair_cnt.items():

        dice = {
            b: 2 * c / (src_cnt[a] + tgt_cnt[b])
            for b, c in cnts.items()
        }

        best = max(dice.values())

        chosen = [
            b
            for b, d in dice.items()
            if d >= ratio * best
        ]

        lex[a] = sorted(
            chosen,
            key=lambda b: avg_pos[b]
        )[:top_k]

    return lex

In [14]:
top_ks = [1, 2, 3, 4, 5]

results_topk = []

for top_k in top_ks:

    lex_tmp = build_dictionary_v2(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=0.625,
        top_k=top_k
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    score_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    results_topk.append({
        "top_k": top_k,
        "ratio": 0.625,
        "val_chrf": score_tmp
    })

pd.DataFrame(results_topk).sort_values(
    "val_chrf",
    ascending=False
)

,top_k,ratio,val_chrf
3,4,0.625,52.469835
2,3,0.625,52.457132
4,5,0.625,52.456240
1,2,0.625,51.368964
0,1,0.625,33.816564


In [15]:
from collections import Counter, defaultdict

def build_dictionary_v3(
    src,
    tgt,
    ratio=0.625,
    top_k=3
):
    src_cnt = Counter()
    tgt_cnt = Counter()
    pair_cnt = defaultdict(Counter)
    pos_sum = defaultdict(float)

    for s, t in zip(src, tgt):
        stoks = set(s.split())
        ttoks = t.split()

        for i, w in enumerate(ttoks):
            pos_sum[w] += i / max(len(ttoks) - 1, 1)

        tset = set(ttoks)

        src_cnt.update(stoks)
        tgt_cnt.update(tset)

        for a in stoks:
            pair_cnt[a].update(tset)

    tgt_freq = Counter(
        w
        for t in tgt
        for w in t.split()
    )

    avg_pos = {
        w: pos_sum[w] / tgt_freq[w]
        for w in tgt_freq
    }

    lex = {}

    for a, cnts in pair_cnt.items():

        dice = {
            b: 2 * c / (src_cnt[a] + tgt_cnt[b])
            for b, c in cnts.items()
        }

        best = max(dice.values())

        chosen = [
            b for b, d in dice.items()
            if d >= ratio * best
        ]

        lex[a] = sorted(
            chosen,
            key=lambda b: avg_pos[b]
        )[:top_k]

    return lex, avg_pos

In [16]:
def translate_reorder(sent, lex, avg_pos):

    out = []

    for w in sent.split():

        if w in lex:
            for tgt_word in lex[w]:
                out.append(
                    (tgt_word, avg_pos.get(tgt_word, 0.5))
                )

        elif w[:1].isupper():
            out.append((w, 0.0))

    if not out:
        return "?"

    out = sorted(
        out,
        key=lambda x: x[1]
    )

    return " ".join(
        word for word, pos in out
    )

In [17]:
lex_v3, avg_pos_v3 = build_dictionary_v3(
    train_fit["src"].tolist(),
    train_fit["translation"].tolist(),
    ratio=0.625,
    top_k=3
)

pred_v3 = [
    translate_reorder(
        s,
        lex_v3,
        avg_pos_v3
    )
    for s in val["src"]
]

score_v3 = chrf(
    pred_v3,
    val["translation"].tolist()
)

bleu_v3 = bleu(
    pred_v3,
    val["translation"].tolist()
)

print("EXP04 chrF:", score_v3)
print("EXP04 BLEU:", bleu_v3)

EXP04 chrF: 45.2508251651294
EXP04 BLEU: 8.411646774861412


In [18]:
compare = val[
    ["src", "translation"]
].copy()

compare["baseline"] = pred_base
compare["reorder"] = pred_v3

display(compare.head(10))

,src,translation,baseline,reorder
0,pupu pasa soyeteka Bodakuni hore sebiri tuwita,những nông dân rất hiền đã đưa nước nóng cho B...,rất hiền nông dân Bodaku nóng nước đưa,nông Bodaku đưa dân hiền rất nước nóng
1,Kekhewu nubeputa ve wada zabeni napuwode hero ...,Kekhewu đã đi dạo và bà cụ sẽ tặng bức tranh c...,Kekhewu đi dạo và chăm chỉ giáo bệnh bà cụ bức...,Kekhewu bà giáo tặng tặng đưa cụ chăm và bức c...
2,heroka zazide bunuto razeri mebe demo huza pas...,những bà cụ mua bốn con gà ở chợ nhưng hai bạn...,bà cụ chợ bốn gà mua nhưng hai hiền bạn làng h...,bà hai bốn giáo bạn cụ gà mua hiền hiền nhưng ...
3,pawu rohoyuka nubepu,những em bé vui vẻ đi dạo,vui vẻ em bé đi dạo,em bé vui vẻ đi dạo
4,hehi wezayo zoyiri wizosu,ba con lợn sẽ tìm cô ấy,ba lợn cô tìm,cô ba lợn tìm
5,himiri pawu rowapi zazide mebesu ve yekodu bun...,học sinh vui vẻ sẽ mua chiếc áo ở chợ và hôm q...,áo vui vẻ sinh chợ mua và hôm qua bốn bác sĩ h...,hôm qua bốn bác sinh sĩ gà vui mua hiền và rất...
6,yekodu Zuguwano pawu nuzuka mobode womuni hehi...,hôm qua những bạn vui vẻ của Zuguwa đã gửi ba ...,hôm qua Zuguwa vui vẻ bạn nhà tôi ba mới mũ gửi,hôm qua Zuguwa ba bạn đưa gửi gửi tôi vui vẻ m...
7,Khangasa husade razata,Khangasa đã bơi ở bờ sông,Khangasa bờ sông bơi,Khangasa bơi bờ sông
8,pawu nuzu hupe nikori gegohine,bạn vui vẻ đang bán năm chiếc mũ,vui vẻ bạn năm mũ bán,năm bạn bán vui vẻ mũ
9,Pigalo napuwode kawata,Pigalo đã ngủ ở bệnh viện,Pigalo bệnh ngủ,Pigalo ngủ bệnh viện


In [19]:
source_pos = {}

for src_word, target_words in lex_v3.items():

    positions = [
        avg_pos_v3[w]
        for w in target_words
        if w in avg_pos_v3
    ]

    if positions:
        source_pos[src_word] = sum(positions) / len(positions)

In [20]:
def translate_block_reorder(sent, lex, source_pos):

    blocks = []

    for i, w in enumerate(sent.split()):

        if w in lex:
            block = lex[w]

            pos = source_pos.get(w, 0.5)

            blocks.append(
                (pos, i, block)
            )

        elif w[:1].isupper():
            blocks.append(
                (0.0, i, [w])
            )

    if not blocks:
        return "?"

    blocks.sort(
        key=lambda x: (x[0], x[1])
    )

    out = []

    for _, _, block in blocks:
        out.extend(block)

    return " ".join(out)

In [21]:
pred_v5 = [
    translate_block_reorder(
        s,
        lex_v3,
        source_pos
    )
    for s in val["src"]
]

score_v5 = chrf(
    pred_v5,
    val["translation"].tolist()
)

bleu_v5 = bleu(
    pred_v5,
    val["translation"].tolist()
)

print("EXP05 chrF:", score_v5)
print("EXP05 BLEU:", bleu_v5)

EXP05 chrF: 51.449522934479845
EXP05 BLEU: 14.187788851947182


In [22]:
compare_v5 = val[
    ["src", "translation"]
].copy()

compare_v5["best_before"] = pred_base
compare_v5["block_reorder"] = pred_v5

display(compare_v5.head(10))

,src,translation,best_before,block_reorder
0,pupu pasa soyeteka Bodakuni hore sebiri tuwita,những nông dân rất hiền đã đưa nước nóng cho B...,rất hiền nông dân Bodaku nóng nước đưa,Bodaku đưa nông dân hiền rất nước nóng
1,Kekhewu nubeputa ve wada zabeni napuwode hero ...,Kekhewu đã đi dạo và bà cụ sẽ tặng bức tranh c...,Kekhewu đi dạo và chăm chỉ giáo bệnh bà cụ bức...,Kekhewu giáo tặng đưa tặng bà cụ và chăm chỉ b...
2,heroka zazide bunuto razeri mebe demo huza pas...,những bà cụ mua bốn con gà ở chợ nhưng hai bạn...,bà cụ chợ bốn gà mua nhưng hai hiền bạn làng h...,hai bốn bạn bà cụ gà mua giáo gặp viên hiền hi...
3,pawu rohoyuka nubepu,những em bé vui vẻ đi dạo,vui vẻ em bé đi dạo,em bé vui vẻ đi dạo
4,hehi wezayo zoyiri wizosu,ba con lợn sẽ tìm cô ấy,ba lợn cô tìm,cô ba lợn tìm
5,himiri pawu rowapi zazide mebesu ve yekodu bun...,học sinh vui vẻ sẽ mua chiếc áo ở chợ và hôm q...,áo vui vẻ sinh chợ mua và hôm qua bốn bác sĩ h...,hôm qua bốn bác sĩ gà mua sinh học hiền và vui...
6,yekodu Zuguwano pawu nuzuka mobode womuni hehi...,hôm qua những bạn vui vẻ của Zuguwa đã gửi ba ...,hôm qua Zuguwa vui vẻ bạn nhà tôi ba mới mũ gửi,hôm qua Zuguwa ba bạn gửi đưa gửi tôi vui vẻ m...
7,Khangasa husade razata,Khangasa đã bơi ở bờ sông,Khangasa bờ sông bơi,Khangasa bơi bờ sông
8,pawu nuzu hupe nikori gegohine,bạn vui vẻ đang bán năm chiếc mũ,vui vẻ bạn năm mũ bán,năm bạn bán vui vẻ mũ
9,Pigalo napuwode kawata,Pigalo đã ngủ ở bệnh viện,Pigalo bệnh ngủ,Pigalo ngủ bệnh viện


In [28]:
ratios = [0.55, 0.575, 0.60, 0.625, 0.65, 0.675]
top_ks = [2, 3, 4, 5]

grid_results = []

for ratio in ratios:
    for top_k in top_ks:

        lex_tmp = build_dictionary_v2(
            train_fit["src"].tolist(),
            train_fit["translation"].tolist(),
            ratio=ratio,
            top_k=top_k
        )

        pred_tmp = [
            translate(s, lex_tmp)
            for s in val["src"]
        ]

        score_tmp = chrf(
            pred_tmp,
            val["translation"].tolist()
        )

        grid_results.append({
            "ratio": ratio,
            "top_k": top_k,
            "val_chrf": score_tmp
        })

grid_df = pd.DataFrame(grid_results)

grid_df.sort_values(
    "val_chrf",
    ascending=False
).head(10)

,ratio,top_k,val_chrf
14,0.625,4,52.469835
13,0.625,3,52.457132
15,0.625,5,52.456240
11,0.600,5,51.949978
10,0.600,4,51.915541
9,0.600,3,51.898131
12,0.625,2,51.368964
3,0.550,5,51.355478
18,0.650,4,51.304874
17,0.650,3,51.294121


In [29]:
lex_best, avg_pos_best = build_dictionary_v3(
    train_fit["src"].tolist(),
    train_fit["translation"].tolist(),
    ratio=0.625,
    top_k=4
)

In [30]:
source_pos_best = {}

for src_word, target_words in lex_best.items():

    positions = [
        avg_pos_best[w]
        for w in target_words
        if w in avg_pos_best
    ]

    if positions:
        source_pos_best[src_word] = sum(positions) / len(positions)

In [31]:
def translate_partial_reorder(
    sent,
    lex,
    source_pos,
    alpha=1.0
):
    words = sent.split()
    n = len(words)

    blocks = []

    for i, w in enumerate(words):

        src_pos = i / max(n - 1, 1)

        if w in lex:
            learned_pos = source_pos.get(w, src_pos)

            final_pos = (
                alpha * src_pos
                + (1 - alpha) * learned_pos
            )

            blocks.append(
                (final_pos, i, lex[w])
            )

        elif w[:1].isupper():
            blocks.append(
                (src_pos, i, [w])
            )

    if not blocks:
        return "?"

    blocks.sort(key=lambda x: (x[0], x[1]))

    out = []

    for _, _, block in blocks:
        out.extend(block)

    return " ".join(out)

In [32]:
alphas = [
    0.0,
    0.25,
    0.50,
    0.75,
    0.90,
    1.0
]

alpha_results = []

for alpha in alphas:

    pred_tmp = [
        translate_partial_reorder(
            s,
            lex_best,
            source_pos_best,
            alpha=alpha
        )
        for s in val["src"]
    ]

    chrf_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    bleu_tmp = bleu(
        pred_tmp,
        val["translation"].tolist()
    )

    alpha_results.append({
        "alpha": alpha,
        "val_chrf": chrf_tmp,
        "val_bleu": bleu_tmp
    })

pd.DataFrame(alpha_results).sort_values(
    "val_chrf",
    ascending=False
)

,alpha,val_chrf,val_bleu
4,0.90,52.469835,15.799184
5,1.00,52.469835,15.799184
1,0.25,52.387864,16.219251
3,0.75,52.022551,14.629853
2,0.50,51.902037,14.478608
0,0.00,51.473990,14.171303


In [33]:
from collections import Counter, defaultdict

def build_bigram_dictionary(
    src,
    tgt,
    ratio=0.625,
    top_k=4,
    min_count=2
):
    phrase_cnt = Counter()
    tgt_cnt = Counter()
    pair_cnt = defaultdict(Counter)

    pos_sum = defaultdict(float)
    tgt_freq = Counter()

    for s, t in zip(src, tgt):
        stoks = s.split()
        ttoks = t.split()

        # vị trí trung bình target word
        for i, w in enumerate(ttoks):
            pos_sum[w] += i / max(len(ttoks) - 1, 1)
            tgt_freq[w] += 1

        # bigram nguồn trong câu
        phrases = {
            tuple(stoks[i:i+2])
            for i in range(len(stoks) - 1)
        }

        tset = set(ttoks)

        phrase_cnt.update(phrases)
        tgt_cnt.update(tset)

        for p in phrases:
            pair_cnt[p].update(tset)

    avg_pos = {
        w: pos_sum[w] / tgt_freq[w]
        for w in tgt_freq
    }

    phrase_lex = {}

    for phrase, cnts in pair_cnt.items():

        if phrase_cnt[phrase] < min_count:
            continue

        dice = {
            b: 2 * c / (phrase_cnt[phrase] + tgt_cnt[b])
            for b, c in cnts.items()
        }

        best = max(dice.values())

        chosen = [
            b for b, d in dice.items()
            if d >= ratio * best
        ]

        phrase_lex[phrase] = sorted(
            chosen,
            key=lambda b: avg_pos[b]
        )[:top_k]

    return phrase_lex

In [34]:
bigram_lex = build_bigram_dictionary(
    train_fit["src"].tolist(),
    train_fit["translation"].tolist(),
    ratio=0.625,
    top_k=4,
    min_count=2
)

print("Số bigram học được:", len(bigram_lex))

Số bigram học được: 10027


In [35]:
def translate_bigram(
    sent,
    unigram_lex,
    bigram_lex
):
    toks = sent.split()
    out = []

    i = 0

    while i < len(toks):

        # thử cụm 2 từ trước
        if i + 1 < len(toks):
            phrase = (toks[i], toks[i + 1])

            # tránh nuốt tên riêng vào phrase
            if (
                phrase in bigram_lex
                and not toks[i][:1].isupper()
                and not toks[i + 1][:1].isupper()
            ):
                out.extend(bigram_lex[phrase])
                i += 2
                continue

        # fallback unigram
        w = toks[i]

        if w in unigram_lex:
            out.extend(unigram_lex[w])

        elif w[:1].isupper():
            out.append(w)

        i += 1

    return " ".join(out) if out else "?"

In [36]:
pred_v8 = [
    translate_bigram(
        s,
        lex_best,
        bigram_lex
    )
    for s in val["src"]
]

score_v8 = chrf(
    pred_v8,
    val["translation"].tolist()
)

bleu_v8 = bleu(
    pred_v8,
    val["translation"].tolist()
)

print("EXP08 chrF:", score_v8)
print("EXP08 BLEU:", bleu_v8)

EXP08 chrF: 32.0122523106486
EXP08 BLEU: 6.807166390734014


In [37]:
compare_v8 = val[
    ["src", "translation"]
].copy()

compare_v8["current_best"] = [
    translate(s, lex_best)
    for s in val["src"]
]

compare_v8["bigram"] = pred_v8

display(compare_v8.head(15))

,src,translation,current_best,bigram
0,pupu pasa soyeteka Bodakuni hore sebiri tuwita,những nông dân rất hiền đã đưa nước nóng cho B...,rất hiền nông dân Bodaku nóng nước đưa,hiền rất nông dân Bodaku nước nóng đưa
1,Kekhewu nubeputa ve wada zabeni napuwode hero ...,Kekhewu đã đi dạo và bà cụ sẽ tặng bức tranh c...,Kekhewu đi dạo và chăm chỉ giáo tặng đưa viên ...,Kekhewu đi dạo Ngokeye Mahopu Gange Gorazo bện...
2,heroka zazide bunuto razeri mebe demo huza pas...,những bà cụ mua bốn con gà ở chợ nhưng hai bạn...,bà cụ chợ bốn gà mua nhưng hai hiền bạn làng h...,Gosobu bốn Wodi hai Mukhoga nhưng bạn hiền Lid...
3,pawu rohoyuka nubepu,những em bé vui vẻ đi dạo,vui vẻ em bé đi dạo,em Dolise Yatu Ngunazi đi dạo
4,hehi wezayo zoyiri wizosu,ba con lợn sẽ tìm cô ấy,ba lợn cô tìm,Solipi lợn Binoki Sula Wekho Boze
5,himiri pawu rowapi zazide mebesu ve yekodu bun...,học sinh vui vẻ sẽ mua chiếc áo ở chợ và hôm q...,áo vui vẻ sinh học chợ mua và hôm qua bốn bác ...,áo sinh vui vẻ chợ mua voi hôm qua bốn Mamizi ...
6,yekodu Zuguwano pawu nuzuka mobode womuni hehi...,hôm qua những bạn vui vẻ của Zuguwa đã gửi ba ...,hôm qua Zuguwa vui vẻ bạn nhà đưa gửi tôi cho ...,hôm qua Zuguwa bạn Dedu vui vẻ Lido Mukabi đạp...
7,Khangasa husade razata,Khangasa đã bơi ở bờ sông,Khangasa bờ sông bơi,Khangasa bơi
8,pawu nuzu hupe nikori gegohine,bạn vui vẻ đang bán năm chiếc mũ,vui vẻ bạn năm mũ bán,bạn vui vẻ mũ bán
9,Pigalo napuwode kawata,Pigalo đã ngủ ở bệnh viện,Pigalo bệnh viện ngủ,Pigalo bệnh viện


In [38]:
def translate_dedup(sent, lex):
    out = []
    seen = set()

    for w in sent.split():

        if w in lex:

            for target_word in lex[w]:

                if target_word not in seen:
                    out.append(target_word)
                    seen.add(target_word)

        elif w[:1].isupper():

            if w not in seen:
                out.append(w)
                seen.add(w)

    return " ".join(out) if out else "?"

In [39]:
lex_best = build_dictionary_v2(
    train_fit["src"].tolist(),
    train_fit["translation"].tolist(),
    ratio=0.625,
    top_k=4
)

In [40]:
pred_v9 = [
    translate_dedup(s, lex_best)
    for s in val["src"]
]

score_v9 = chrf(
    pred_v9,
    val["translation"].tolist()
)

bleu_v9 = bleu(
    pred_v9,
    val["translation"].tolist()
)

print("EXP09 chrF:", score_v9)
print("EXP09 BLEU:", bleu_v9)

EXP09 chrF: 51.755155606292796
EXP09 BLEU: 14.939592442166695


In [41]:
pred_best = [
    translate(s, lex_best)
    for s in val["src"]
]

changed = sum(
    a != b
    for a, b in zip(pred_best, pred_v9)
)

print("Số câu bị thay đổi:", changed)
print("Tổng validation:", len(val))

Số câu bị thay đổi: 593
Tổng validation: 3981


In [42]:
compare_v9 = val[
    ["src", "translation"]
].copy()

compare_v9["current_best"] = pred_best
compare_v9["dedup"] = pred_v9

display(
    compare_v9[
        compare_v9["current_best"]
        != compare_v9["dedup"]
    ].head(15)
)

,src,translation,current_best,dedup
1,Kekhewu nubeputa ve wada zabeni napuwode hero ...,Kekhewu đã đi dạo và bà cụ sẽ tặng bức tranh c...,Kekhewu đi dạo và chăm chỉ giáo tặng đưa viên ...,Kekhewu đi dạo và chăm chỉ giáo tặng đưa viên ...
2,heroka zazide bunuto razeri mebe demo huza pas...,những bà cụ mua bốn con gà ở chợ nhưng hai bạn...,bà cụ chợ bốn gà mua nhưng hai hiền bạn làng h...,bà cụ chợ bốn gà mua nhưng hai hiền bạn làng g...
6,yekodu Zuguwano pawu nuzuka mobode womuni hehi...,hôm qua những bạn vui vẻ của Zuguwa đã gửi ba ...,hôm qua Zuguwa vui vẻ bạn nhà đưa gửi tôi cho ...,hôm qua Zuguwa vui vẻ bạn nhà đưa gửi tôi cho ...
31,nugu kawata ve yusebo siku mawohu mobode kawane,anh ấy đã ngủ và bây giờ con ngựa trắng đang n...,anh ấy ngủ và bây giờ trắng ngựa nhà ngủ,anh ấy ngủ và bây giờ trắng ngựa nhà
32,robi napuwode zadokani pipori bazobeta,họ đã tặng phở cho những bác sĩ ở bệnh viện,họ bệnh viện Wuzopi bác tặng Karime phở tặng,họ bệnh viện Wuzopi bác tặng Karime phở
35,nuzuka karude kiboni hore hutamori bazobe ve z...,những bạn tặng cơm nóng cho mẹ ở trường học và...,bạn học trường mẹ nóng cơm tặng và cô ấy học t...,bạn học trường mẹ nóng cơm tặng và cô ấy đẹp m...
40,Sumo husade resota demo Godigu husade resomata,Sumo đã câu cá ở bờ sông nhưng Godigu chưa câu...,Sumo bờ sông câu cá nhưng Godigu bờ sông chưa ...,Sumo bờ sông câu cá nhưng Godigu chưa
43,nemuke deku rowapi husade nubepusu kara nemuke...,ngày mai chúng tôi sẽ câu cá ở bờ sông vì ngày...,ngày mai già sinh học bờ sông đi dạo vì ngày m...,ngày mai già sinh học bờ sông đi dạo vì chúng ...
51,deku goruka husade resota kara deku godegeno h...,hai con chó già của người bán hàng già đang há...,già anh trai bờ sông câu cá vì già người bán c...,già anh trai bờ sông câu cá vì người bán của h...
52,powe husade raza ve husade rewuzeka razata,con chó bơi ở bờ sông và những con dê đã bơi ở...,chó bờ sông bơi và bờ sông dê bơi,chó bờ sông bơi và dê


In [43]:
src_n_trans = train.groupby("src")["translation"].nunique()

print("Số src lặp:", (train["src"].value_counts() > 1).sum())
print("Src có >1 translation:", (src_n_trans > 1).sum())

Số src lặp: 337
Src có >1 translation: 0


In [44]:
test_public = pd.read_csv("data/test_public/data.csv")

train_src_set = set(train["src"])

exact_overlap = test_public["src"].isin(train_src_set)

print("Test size:", len(test_public))
print("Exact train-test src overlap:", exact_overlap.sum())
print("Overlap %:", exact_overlap.mean() * 100)

Test size: 1000
Exact train-test src overlap: 0
Overlap %: 0.0


In [45]:
sentence_lex = (
    train
    .groupby("src")["translation"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

print("Số câu nhớ được:", len(sentence_lex))

Số câu nhớ được: 19551


In [46]:
def translate_hybrid(sent, sentence_lex, word_lex):

    if sent in sentence_lex:
        return sentence_lex[sent]

    return translate(sent, word_lex)

In [47]:
import math
from collections import Counter, defaultdict

def build_dictionary_assoc(
    src,
    tgt,
    ratio=0.625,
    top_k=4,
    score_type="dice"
):
    src_cnt = Counter()
    tgt_cnt = Counter()
    pair_cnt = defaultdict(Counter)
    pos_sum = defaultdict(float)

    for s, t in zip(src, tgt):
        stoks = set(s.split())
        ttoks = t.split()

        for i, w in enumerate(ttoks):
            pos_sum[w] += i / max(len(ttoks) - 1, 1)

        tset = set(ttoks)

        src_cnt.update(stoks)
        tgt_cnt.update(tset)

        for a in stoks:
            pair_cnt[a].update(tset)

    tgt_freq = Counter(
        w for t in tgt for w in t.split()
    )

    avg_pos = {
        w: pos_sum[w] / tgt_freq[w]
        for w in tgt_freq
    }

    lex = {}

    for a, cnts in pair_cnt.items():

        scores = {}

        for b, c in cnts.items():
            A = src_cnt[a]
            B = tgt_cnt[b]

            if score_type == "dice":
                score = 2 * c / (A + B)

            elif score_type == "ochiai":
                score = c / math.sqrt(A * B)

            elif score_type == "jaccard":
                score = c / (A + B - c)

            scores[b] = score

        best = max(scores.values())

        chosen = [
            b for b, score in scores.items()
            if score >= ratio * best
        ]

        lex[a] = sorted(
            chosen,
            key=lambda b: avg_pos[b]
        )[:top_k]

    return lex

In [48]:
score_types = [
    "dice",
    "ochiai",
    "jaccard"
]

assoc_results = []

for score_type in score_types:

    lex_tmp = build_dictionary_assoc(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=0.625,
        top_k=4,
        score_type=score_type
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    chrf_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    bleu_tmp = bleu(
        pred_tmp,
        val["translation"].tolist()
    )

    assoc_results.append({
        "score_type": score_type,
        "val_chrf": chrf_tmp,
        "val_bleu": bleu_tmp
    })

pd.DataFrame(assoc_results).sort_values(
    "val_chrf",
    ascending=False
)

,score_type,val_chrf,val_bleu
1,ochiai,53.002297,16.936064
0,dice,52.469835,15.799184
2,jaccard,48.336528,11.903957


In [49]:
ratios_ochiai = [
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.625,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]

ochiai_ratio_results = []

for ratio in ratios_ochiai:

    lex_tmp = build_dictionary_assoc(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=ratio,
        top_k=4,
        score_type="ochiai"
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    chrf_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    bleu_tmp = bleu(
        pred_tmp,
        val["translation"].tolist()
    )

    ochiai_ratio_results.append({
        "ratio": ratio,
        "val_chrf": chrf_tmp,
        "val_bleu": bleu_tmp
    })

ochiai_ratio_df = pd.DataFrame(
    ochiai_ratio_results
)

ochiai_ratio_df.sort_values(
    "val_chrf",
    ascending=False
)

,ratio,val_chrf,val_bleu
1,0.450,54.878473,18.271920
0,0.400,54.768127,16.992014
2,0.500,53.294134,15.046223
4,0.600,53.132700,17.061143
5,0.625,53.002297,16.936064
3,0.550,52.928284,15.419204
7,0.700,52.603779,15.020254
8,0.750,52.194837,14.375498
6,0.650,52.179998,16.727612
9,0.800,49.057531,11.965318


In [50]:
ratios_fine_ochiai = [
    0.40,
    0.425,
    0.45,
    0.475,
    0.50
]

fine_results = []

for ratio in ratios_fine_ochiai:

    lex_tmp = build_dictionary_assoc(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=ratio,
        top_k=4,
        score_type="ochiai"
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    chrf_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    bleu_tmp = bleu(
        pred_tmp,
        val["translation"].tolist()
    )

    fine_results.append({
        "ratio": ratio,
        "val_chrf": chrf_tmp,
        "val_bleu": bleu_tmp
    })

pd.DataFrame(fine_results).sort_values(
    "val_chrf",
    ascending=False
)

,ratio,val_chrf,val_bleu
1,0.425,55.645052,19.249303
2,0.450,54.878473,18.271920
0,0.400,54.768127,16.992014
3,0.475,54.290918,17.309450
4,0.500,53.294134,15.046223


In [51]:
top_ks_ochiai = [2, 3, 4, 5, 6]

topk_results = []

for top_k in top_ks_ochiai:

    lex_tmp = build_dictionary_assoc(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=0.425,
        top_k=top_k,
        score_type="ochiai"
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    chrf_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    bleu_tmp = bleu(
        pred_tmp,
        val["translation"].tolist()
    )

    topk_results.append({
        "top_k": top_k,
        "ratio": 0.425,
        "val_chrf": chrf_tmp,
        "val_bleu": bleu_tmp
    })

pd.DataFrame(topk_results).sort_values(
    "val_chrf",
    ascending=False
)

,top_k,ratio,val_chrf,val_bleu
4,6,0.425,56.043249,20.623976
3,5,0.425,55.994379,19.484179
2,4,0.425,55.645052,19.249303
1,3,0.425,55.344107,18.943745
0,2,0.425,55.322113,18.655288


In [52]:
top_ks_more = [6, 7, 8, 9, 10]

topk_more_results = []

for top_k in top_ks_more:

    lex_tmp = build_dictionary_assoc(
        train_fit["src"].tolist(),
        train_fit["translation"].tolist(),
        ratio=0.425,
        top_k=top_k,
        score_type="ochiai"
    )

    pred_tmp = [
        translate(s, lex_tmp)
        for s in val["src"]
    ]

    chrf_tmp = chrf(
        pred_tmp,
        val["translation"].tolist()
    )

    bleu_tmp = bleu(
        pred_tmp,
        val["translation"].tolist()
    )

    topk_more_results.append({
        "top_k": top_k,
        "val_chrf": chrf_tmp,
        "val_bleu": bleu_tmp
    })

pd.DataFrame(topk_more_results).sort_values(
    "val_chrf",
    ascending=False
)

,top_k,val_chrf,val_bleu
0,6,56.043249,20.623976
1,7,56.013912,20.595825
3,9,56.009589,20.587201
2,8,56.008927,20.588570
4,10,56.005633,20.585359


In [54]:
from sklearn.model_selection import GroupShuffleSplit

def evaluate_seed(seed):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=seed
    )

    train_idx, val_idx = next(
        splitter.split(train, groups=train["src"])
    )

    tr_seed = train.iloc[train_idx].reset_index(drop=True)
    va_seed = train.iloc[val_idx].reset_index(drop=True)

    # Baseline gốc
    lex_base_seed = build_dictionary(
        tr_seed["src"].tolist(),
        tr_seed["translation"].tolist(),
        ratio=0.9
    )

    pred_base_seed = [
        translate(s, lex_base_seed)
        for s in va_seed["src"]
    ]

    baseline_chrf = chrf(
        pred_base_seed,
        va_seed["translation"].tolist()
    )

    # Current best
    lex_best_seed = build_dictionary_assoc(
        tr_seed["src"].tolist(),
        tr_seed["translation"].tolist(),
        ratio=0.425,
        top_k=6,
        score_type="ochiai"
    )

    pred_best_seed = [
        translate(s, lex_best_seed)
        for s in va_seed["src"]
    ]

    best_chrf = chrf(
        pred_best_seed,
        va_seed["translation"].tolist()
    )

    return {
        "seed": seed,
        "baseline_chrf": baseline_chrf,
        "best_chrf": best_chrf,
        "gain": best_chrf - baseline_chrf
    }


In [55]:
seeds = [42, 123, 2026, 7, 99]

seed_results = [
    evaluate_seed(seed)
    for seed in seeds
]

seed_df = pd.DataFrame(seed_results)

display(seed_df)

print(
    "Baseline mean:",
    seed_df["baseline_chrf"].mean()
)

print(
    "Best mean:",
    seed_df["best_chrf"].mean()
)

print(
    "Mean gain:",
    seed_df["gain"].mean()
)

print(
    "Best std:",
    seed_df["best_chrf"].std()
)

,seed,baseline_chrf,best_chrf,gain
0,42,44.993686,56.043249,11.049564
1,123,44.848355,55.937389,11.089034
2,2026,44.732190,55.697629,10.965439
3,7,44.687943,55.999235,11.311292
4,99,44.687107,55.947185,11.260078


Baseline mean: 44.789856030540015
Best mean: 55.924937465712404
Mean gain: 11.13508143517238
Best std: 0.1340120839047471
